# Entrega 3 · Formulación LP/MILP con tres agentes

El Analista extrae requisitos, el Formulador redacta una formulación y el Verificador la revisa con una auditoría determinista opcional. El caso de centros oftalmológicos permite seguir sus mensajes y contrastar las respuestas de las tres entregas. **FOM-5 v2** puntúa después de la generación, como evaluador externo.

El cuaderno tiene dos recorridos. Las secciones 1–4 muestran el caso diagnóstico y sus resultados archivados; `RUN_LIVE_DIAGNOSTIC=True` permite generar otra respuesta. La sección 6 ejecuta **cinco configuraciones sobre los mismos 15 casos**, una configuración por cuenta T4, y guarda cada caso en Drive. La puntuación de los 15 se inicia expresamente tras reunir las cinco carpetas.

## 1. Repositorio y dependencias

Conecta una GPU **T4** y ejecuta las tres celdas de esta sección. La primera descarga `main` y comprueba `v3-freeze`; la segunda instala dependencias; la tercera carga Qwen2.5-1.5B-Instruct y el adapter QLoRA público. Si Colab pide reiniciar después de instalar PyTorch, reinicia y repite la sección. Cada cuenta monta su Drive, donde la carpeta compartida de resultados debe verse en `Mi unidad` con el nombre `GenAI_Entrega3_v3freeze` (carpeta original o acceso directo).

In [ ]:
from pathlib import Path
import subprocess, sys

STAGE = 'oficial'
SYNC = False  # Los resultados largos se guardan en Drive.
DRIVE_FOLDER_NAME = 'GenAI_Entrega3_v3freeze'  # Carpeta compartida o acceso directo en Mi unidad.
REPO = Path('/content/GenAI-Optimization')
URL = 'https://github.com/Jose21531/GenAI-Optimization.git'

def git(*args):
    return subprocess.run(['git', '-C', str(REPO), *args], check=True,
                          capture_output=True, text=True).stdout.strip()

if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', URL, str(REPO)], check=True)
git('fetch', '-q', '--tags', 'origin')
git('checkout', '-q', '-B', 'd3-main', 'origin/main')
assert subprocess.run(['git', '-C', str(REPO), 'rev-parse', '-q', '--verify',
                       'refs/tags/v3-freeze'], capture_output=True).returncode == 0
D3 = REPO / 'Deliverable 3'
RESULTS = D3 / 'results'
print('Código:', git('rev-parse', '--short', 'HEAD'), '| configuración:', 'v3-freeze')


In [ ]:
# El perfil de PyTorch también es el que exige el juez FOM-5 v2.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(D3 / 'eval/requirements-gpu.txt'), 'peft==0.15.2'], check=True)
print('Dependencias instaladas. Reinicia el entorno si Colab lo solicita.')

In [ ]:
import hashlib, json, tempfile
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Conecta una GPU T4.'
assert str(torch.__version__) == '2.6.0+cu124', 'Reinicia el entorno y ejecuta desde la primera celda.'
sys.path.insert(0, str(D3))
from model import ADAPTER_SHA256, extract_adapter, load_qwen, sha256_file
from run import load_cases, load_config, run_variant, _identity

ADAPTER_ZIP = REPO / 'Deliverable 2/adapter_qlora_qwen25_final.zip'
assert sha256_file(ADAPTER_ZIP) == ADAPTER_SHA256
ADAPTER_TEMP = tempfile.TemporaryDirectory(prefix='d3-adapter-')
ADAPTER_DIR = extract_adapter(ADAPTER_ZIP, Path(ADAPTER_TEMP.name) / 'adapter')
llm = load_qwen(ADAPTER_DIR)
config, _ = load_config('diagnostic')
print('T4, modelo base y adapter público preparados.')

In [ ]:
from gitsync import checkpoint
from eval.recovery import recover_jsonl_tail

def sync(paths, message):
    if SYNC:
        print('Subido' if checkpoint(paths, message) else 'Pendiente de subir', '-', message)

def recover_tails(folder):
    # Una sesión cortada a mitad de una escritura deja una línea incompleta.
    for path in Path(folder).rglob('*.jsonl'):
        data = path.read_bytes()
        if data and not data.endswith(b'\n'):
            print(recover_jsonl_tail(path))

def run_synced(command, paths, message, every=300):
    """Ejecuta un proceso largo y sube `paths` cada `every` segundos y al terminar."""
    process = subprocess.Popen(command)
    while True:
        try:
            process.wait(timeout=every)
            break
        except subprocess.TimeoutExpired:
            sync(paths, message)
    sync(paths, message)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)

## 2. Caso de Entrega 1: mensajes de los agentes

La vista inicial carga la corrida diagnóstica **archivada** del caso de centros oftalmológicos. Para grabar la ejecución real, fija `RUN_LIVE_DIAGNOSTIC=True` después de preparar la T4 en la sección 1 y ejecuta la celda durante la grabación. La respuesta y la traza de esa nueva corrida se guardan en `MyDrive/GenAI_Entrega3_v3freeze/diagnostic/<fecha>/`; después podrás exportarlas desde la celda de transferencia. La entrada de los agentes contiene solo el enunciado; la referencia se abre tras generar la respuesta.

In [ ]:
import json, hashlib
from datetime import datetime, timezone
from types import SimpleNamespace
RUN_LIVE_DIAGNOSTIC = False

def repo_text(relative):
    path = REPO / relative
    if path.exists():
        return path.read_text(encoding='utf-8')
    return subprocess.check_output(
        ['git', '-C', str(REPO), 'show', f'origin/main:{relative}']).decode('utf-8')

cases = {row['id']: row for row in (json.loads(line) for line in
         repo_text('Deliverable 3/data/diagnostic_oftalmo_input.jsonl').splitlines())}
if RUN_LIVE_DIAGNOSTIC:
    from google.colab import drive
    drive.mount('/content/drive')
    DEMO_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    DRIVE_ROOT = Path('/content/drive/MyDrive') / DRIVE_FOLDER_NAME
    assert DRIVE_ROOT.is_dir(), f'Agrega la carpeta compartida a Mi unidad como {DRIVE_FOLDER_NAME}.'
    DEMO_DIR = DRIVE_ROOT / 'diagnostic' / DEMO_ID
    results = run_variant(llm, config, cases, 'full', DEMO_DIR / 'full',
                          DEMO_DIR / 'traces', mode='diagnostic',
                          identity=_identity('diagnostic', cases, ADAPTER_ZIP))
    result = results['diagnostic_oftalmo']
    trace_path = next((DEMO_DIR / 'traces').glob('*.jsonl'))
    events = [json.loads(line) for line in trace_path.read_text(encoding='utf-8').splitlines()]
    sync([DEMO_DIR], f'Caso diagnóstico {DEMO_ID}')
else:
    DEMO_DIR = RESULTS / 'diagnostic/initial'
    row = json.loads(repo_text('Deliverable 3/results/diagnostic/initial/full/outputs.jsonl'))
    result = SimpleNamespace(**row)
    trace_rel = ('Deliverable 3/traces/diagnostic/initial/full/'
                 'diagnostic-full-diagnostic_oftalmo-32780b0f.jsonl')
    events = [json.loads(line) for line in repo_text(trace_rel).splitlines()]
    print('Corrida archivada; actívala arriba para generar una respuesta nueva.')
print('Estado:', result.status, '| tokens:', result.tokens_total,
      '| revisiones:', result.rounds, '| herramientas:', result.tool_calls)

### Lectura de la traza

Los bloques siguen el orden efectivo de la corrida. Las decisiones, el plan, el texto `candidate`, las observaciones y la respuesta final permanecen separados en el JSONL completo.

In [ ]:
from IPython.display import HTML, Markdown, display
from html import escape

for event in events:
    actor, kind = event['actor'], event['event']
    if actor == 'analista' and kind == 'decision':
        artifact = event['output']['artifact']
        spec = artifact.get('spec') or {}
        display(Markdown('### Analista · ' + artifact['action']))
        if spec:
            display(Markdown('**Objetivo:** ' + spec['objective']['description']))
            for req in spec['requirements']:
                evidence = (req.get('evidence') or {}).get('text', '')
                display(Markdown(f"- **{req['id']}** {req['text']}  \n  Evidencia: «{evidence}»"))
    elif actor == 'formulador' and kind == 'decision':
        artifact = event['output']['artifact']
        display(Markdown('### Formulador · ' + artifact['action']))
        if artifact.get('plan'):
            display(Markdown('**Plan requisito → ecuación**'))
            for item in artifact['plan']:
                display(Markdown(f"- {item['requirement_id']}: `{item['equation']}`"))
        draft = artifact.get('draft')
        if draft:
            display(HTML('<pre style="white-space:pre-wrap;max-height:270px;overflow:auto">'
                         + escape(draft['raw_text']) + '</pre>'))
    elif actor == 'audit_model' and kind == 'herramienta':
        findings = (event.get('output') or {}).get('findings', [])
        display(Markdown(f'### audit_model · {len(findings)} hallazgos'))
        for finding in findings[:8]:
            display(Markdown(f"- **{finding['check']}**: {finding['message']}"))
    elif actor == 'verificador' and kind == 'decision':
        artifact = event['output']['artifact']
        decision = artifact.get('decision') or {}
        display(Markdown('### Verificador · ' + (decision.get('action') or artifact['action'])))
        coverage = artifact.get('coverage') or {}
        if coverage:
            display(Markdown(f"Cobertura declarada: {len(coverage.get('links', []))} vínculos a "
                             f"{len(coverage.get('elements', []))} partes del borrador."))
        for issue in decision.get('issues', []):
            display(Markdown('- ' + issue['description']))
display(Markdown(f'### Salida final · {result.status}'))

## 3. Contraste de las tres entregas

Las respuestas baseline y QLoRA proceden de los CSV evaluados de Entrega 2. Con `RUN_LIVE_DIAGNOSTIC=False`, la tercera respuesta y su puntuación proceden de la corrida archivada de Entrega 3. Si ejecutas el caso en vivo, la nueva respuesta se puntúa solo al activar el juez en la sección siguiente.

In [ ]:
from eval.diagnostic import archived_diagnostic, diagnostic_gold

gold, archived = diagnostic_gold()
assert gold['problem'] == cases['diagnostic_oftalmo']['problem']
def card(title, candidate, color, note):
    return (f'<div style="flex:1;min-width:270px;border:1px solid {color};border-radius:9px;'
            f'padding:12px;background:white"><b style="color:{color}">{escape(title)}</b>'
            f'<br><small>{escape(note)}</small><pre style="white-space:pre-wrap;'
            f'max-height:340px;overflow:auto;font-size:12px">{escape(candidate)}</pre></div>')

display(Markdown('### Formulación de referencia'))
display(Markdown(gold['reference_model']))
display(HTML('<div style="display:flex;gap:10px;flex-wrap:wrap">' +
    card('Baseline', archived['baseline']['candidate'], '#5a7184',
         f"FOM-5 v2 archivado: {float(archived['baseline']['score_fom5_v2']):.3f}/5") +
    card('Entrega 2 · QLoRA', archived['QLoRA']['candidate'], '#087f8c',
         f"FOM-5 v2 archivado: {float(archived['QLoRA']['score_fom5_v2']):.3f}/5") +
    card('Entrega 3 · agentes', result.final_text, '#9a5125',
         'Corrida en vivo; requiere juicio nuevo' if RUN_LIVE_DIAGNOSTIC else
         'Corrida archivada; FOM-5 v2: 2.600/5') + '</div>'))
if not RUN_LIVE_DIAGNOSTIC:
    display(Markdown("""### Lectura matemática del caso archivado

| Condición de la referencia | Baseline | QLoRA · Entrega 2 | Agentes · Entrega 3 |
|---|---|---|---|
| Población $p_j$ pondera la distancia | Sí | No | Sí |
| Dos aperturas con $\sum_i y_i=2$ | No | Sí | No: índices incorrectos |
| Asignación $x_{ij}\le y_i$ | No | Sí | No |
| Cada sector se asigna una vez | Contradicción con otra ecuación | Sí | Sí, con ecuaciones adicionales incorrectas |

La tabla resume la lectura de estas tres respuestas exactas; el puntaje FOM-5 v2 aparece en la sección siguiente."""))
else:
    display(Markdown('Revisa en la respuesta nueva: $p_j$ en el objetivo, '
                     '$\sum_i y_i=2$, $x_{ij}\le y_i$ y asignación única.'))

## 4. FOM-5 v2 del caso diagnóstico

El resultado archivado pertenece a la respuesta exacta mostrada arriba. Si ejecutas una respuesta nueva, activa `RUN_DIAGNOSTIC_JUDGE=True` para que Qwen3-14B calibre y puntúe ese texto; se guarda el hash de la respuesta. El caso diagnóstico queda fuera de la muestra de 15 problemas de la sección 6.

In [ ]:
RUN_DIAGNOSTIC_JUDGE = False
score_path = DEMO_DIR / 'fom5_v2_diagnostic.json'
if RUN_DIAGNOSTIC_JUDGE:
    import gc
    if 'llm' in globals():
        del llm
    gc.collect()
    torch.cuda.empty_cache()
    from eval.diagnostic import evaluate_diagnostic
    score = evaluate_diagnostic(result.final_text, score_path)
    print('FOM-5 v2:', score['baseline_archived'], '→', score['qlora_archived'],
          '→', score['scores']['score_fom5_v2'])
    print('Juicio y calibración:', score_path)
    sync([DEMO_DIR], f'Juicio del caso diagnóstico {DEMO_ID}')
elif score_path.exists():
    score = json.loads(score_path.read_text(encoding='utf-8'))
    assert score['candidate_sha256'] == hashlib.sha256(result.final_text.encode()).hexdigest()
    print('FOM-5 v2:', score['baseline_archived'], '→', score['qlora_archived'],
          '→', score['scores']['score_fom5_v2'])
else:
    archived_score = json.loads(repo_text('Deliverable 3/results/diagnostic/initial/fom5_v2_diagnostic.json'))
    if archived_score['candidate_sha256'] == hashlib.sha256(result.final_text.encode()).hexdigest():
        print('FOM-5 v2 archivado: baseline 0.650/5 → QLoRA 3.850/5 → agentes 2.600/5')
    else:
        print('Esta respuesta nueva aún no tiene puntuación. Activa RUN_DIAGNOSTIC_JUDGE.')

## 5. Piloto de desarrollo

La configuración congelada procede de ocho casos de desarrollo. El Verificador con el adapter apagado tuvo **un** fallo de formato; con el adapter encendido, **seis**. El adapter quedó activo solo en el Formulador. Las salidas y trazas del piloto están archivadas en `results/pilot/` y `traces/pilot/`.

In [ ]:
from pathlib import Path
for label in ('verifier_off', 'verifier_on'):
    folder = D3 / 'results/pilot' / label
    files = list(folder.rglob('outputs.jsonl')) if folder.exists() else []
    count = sum(len(p.read_text(encoding='utf-8').splitlines()) for p in files)
    print(label, '| salidas archivadas:', count)
print('Configuración congelada: Verificador sin adapter; Formulador con QLoRA.')


## 6. Generación por configuración y tramo

Las cinco variantes se ejecutan sobre **los mismos 15 enunciados** (`opt_01` a `opt_15`). Selecciona **1 = `full`**, **2 = `no_evidence`**, **3 = `no_plan`**, **4 = `no_audit`** o **5 = `no_review`**. `FIRST_CASE` y `LAST_CASE` delimitan un tramo inclusivo de 1–15. Los puntajes archivados de baseline y QLoRA se compararán en esos mismos IDs.

Todas las cuentas usan la [misma carpeta compartida](https://drive.google.com/drive/folders/1paO1lnuqA1tNWOl9_T07WkH_JzGL66wD?usp=sharing). Antes de ejecutar, cada cuenta debe verla en `Mi unidad` como `GenAI_Entrega3_v3freeze`; en las cuentas invitadas, añade un acceso directo con ese nombre. La celda imprime la ruta resuelta para comprobarla. Una variante por cuenta T4 y **nunca dos sesiones de la misma variante a la vez**. Cada caso y su traza se escriben al terminar; al repetir la celda, se omiten los IDs completos. La muestra de Entrega 3 es de 15 casos, no de los 30 completos.

In [ ]:
RUN_VARIANT = False
CONFIGURATION = 1      # 1: full; 2: no_evidence; 3: no_plan; 4: no_audit; 5: no_review
FIRST_CASE = 1         # Índice inclusivo, entre 1 y 15.
LAST_CASE = 15         # Índice inclusivo, entre 1 y 15.

VARIANTS = {1: 'full', 2: 'no_evidence', 3: 'no_plan',
            4: 'no_audit', 5: 'no_review'}
assert CONFIGURATION in VARIANTS
variant = VARIANTS[CONFIGURATION]
limit = 15
last = LAST_CASE
assert 1 <= FIRST_CASE <= last <= limit
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive') / DRIVE_FOLDER_NAME
assert DRIVE_ROOT.is_dir(), f'Agrega la carpeta compartida a Mi unidad como {DRIVE_FOLDER_NAME}.'
OUTPUT_ROOT = DRIVE_ROOT / 'results'
TRACE_ROOT = DRIVE_ROOT / 'traces'
print('Carpeta de checkpoints:', DRIVE_ROOT.resolve())

if RUN_VARIANT:
    assert STAGE == 'oficial', 'Fija STAGE="oficial" en la primera celda.'
    import torch, tempfile
    from model import ADAPTER_SHA256, extract_adapter, load_qwen, sha256_file
    from run import load_cases, load_config, run_variant, _identity
    from eval.artifacts import read_jsonl
    from eval.artifacts import sha256
    from eval.recovery import recover_jsonl_tail
    assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
    ADAPTER_ZIP = REPO / 'Deliverable 2/adapter_qlora_qwen25_final.zip'
    assert sha256_file(ADAPTER_ZIP) == ADAPTER_SHA256
    if 'llm' not in globals():
        ADAPTER_TEMP = tempfile.TemporaryDirectory(prefix='d3-adapter-')
        adapter = extract_adapter(ADAPTER_ZIP, Path(ADAPTER_TEMP.name) / 'adapter')
        llm = load_qwen(adapter)
    all_cases = load_cases('official')
    frozen_config, _ = load_config('official')
    identity = _identity('official', all_cases, ADAPTER_ZIP)
    output_dir, trace_dir = OUTPUT_ROOT / variant, TRACE_ROOT / variant
    predictions = output_dir / 'outputs.jsonl'
    if predictions.exists() and not predictions.read_bytes().endswith(b'\n'):
        print(recover_jsonl_tail(predictions))
    existing = read_jsonl(predictions) if predictions.exists() else []
    existing_ids = {row['problem_id'] for row in existing}
    assert all(identifier in all_cases and int(identifier[-2:]) <= limit
               for identifier in existing_ids), 'Checkpoint con IDs fuera de este diseño.'
    requested = {f'opt_{number:02}' for number in range(FIRST_CASE, last + 1)}
    work_cases = {key: all_cases[key] for key in sorted(existing_ids | requested)}
    def save_case(step):
        if SYNC:
            from gitsync import checkpoint
            ok = checkpoint([output_dir, trace_dir], f'Entrega 3 {variant}: {step}')
            print('Checkpoint GitHub:', 'subido' if ok else 'pendiente', step, flush=True)
    completed = run_variant(llm, frozen_config, work_cases, variant, output_dir,
                            trace_dir, mode='official_segment', identity=identity,
                            after_case=save_case if SYNC else None)
    subset_ids = [f'opt_{number:02}' for number in range(1, 16)]
    if set(completed) == set(subset_ids):
        subset_manifest = {'design': 'five_variants_first15', 'variant': variant,
                           'ids': subset_ids, 'outputs_sha256': sha256(predictions),
                           'generation_identity': identity}
        manifest_path = output_dir / 'subset_manifest.json'
        manifest_path.write_text(json.dumps(subset_manifest, ensure_ascii=False, indent=2) + '\n',
                                 encoding='utf-8')
        save_case('manifiesto')
    print(f'{variant}: {len(completed)}/{limit} casos guardados en {output_dir}')
else:
    print(f'Listo para {variant}, casos {FIRST_CASE} a {last}. Activa RUN_VARIANT.')


In [ ]:
# Estado local de los checkpoints de esta cuenta.
for number, name in VARIANTS.items():
    path = OUTPUT_ROOT / name / 'outputs.jsonl'
    count = len(path.read_text(encoding='utf-8').splitlines()) if path.exists() else 0
    target = 15
    print(f'{number}. {name}: {count}/{target}')


### Reunir variantes de varias cuentas

Después de completar una variante, ejecuta la celda siguiente con `EXPORT_VARIANT=True`: descarga un ZIP con `results/<variante>` y `traces/<variante>`. En la cuenta que hará la evaluación, ejecuta la misma celda con `IMPORT_ARCHIVES=True` y selecciona los ZIP terminados. La importación comprueba que no sobrescriba archivos distintos. Las cinco carpetas deben quedar bajo el mismo `MyDrive/GenAI_Entrega3_v3freeze/` antes de activar el juez.

In [ ]:
EXPORT_VARIANT = False
IMPORT_ARCHIVES = False
EXPORT_DIAGNOSTIC = False

if EXPORT_VARIANT or IMPORT_ARCHIVES or EXPORT_DIAGNOSTIC:
    from google.colab import drive, files
    from zipfile import ZipFile, ZIP_DEFLATED
    from io import BytesIO
    import hashlib
    drive.mount('/content/drive')
    store = OUTPUT_ROOT.parent
    store.mkdir(parents=True, exist_ok=True)

if EXPORT_VARIANT:
    archive = Path('/content') / f'entrega3_{variant}_15.zip'
    with ZipFile(archive, 'w', ZIP_DEFLATED) as zipped:
        for folder in (OUTPUT_ROOT / variant, TRACE_ROOT / variant):
            for source in sorted(folder.rglob('*')):
                if source.is_file():
                    zipped.write(source, source.relative_to(store))
    print('Exportado:', archive, '| bytes:', archive.stat().st_size)
    files.download(str(archive))

if IMPORT_ARCHIVES:
    uploaded = files.upload()
    for name, data in uploaded.items():
        with ZipFile(BytesIO(data)) as zipped:
            members = zipped.infolist()
            for item in members:
                parts = Path(item.filename).parts
                assert len(parts) >= 3 and parts[0] in {'results', 'traces'}
                assert parts[1] in VARIANTS.values() and '..' not in parts
                destination = store.joinpath(*parts)
                if item.is_dir():
                    continue
                content = zipped.read(item)
                destination.parent.mkdir(parents=True, exist_ok=True)
                if destination.exists():
                    assert destination.read_bytes() == content, f'Archivo distinto: {destination}'
                else:
                    destination.write_bytes(content)
        print('Importado:', name)

if EXPORT_DIAGNOSTIC:
    assert RUN_LIVE_DIAGNOSTIC and DEMO_DIR.is_dir()
    archive = Path('/content') / f'entrega3_diagnostic_{DEMO_ID}.zip'
    with ZipFile(archive, 'w', ZIP_DEFLATED) as zipped:
        for item in sorted(DEMO_DIR.rglob('*')):
            if item.is_file():
                zipped.write(item, Path('diagnostic') / DEMO_ID / item.relative_to(DEMO_DIR))
    print('Diagnóstico exportado:', archive)
    files.download(str(archive))


In [ ]:
RUN_EVALUATE_15 = False
if RUN_EVALUATE_15:
    from google.colab import drive
    drive.mount('/content/drive')
    import gc
    if 'llm' in globals():
        del llm
        gc.collect()
        torch.cuda.empty_cache()
    sources = [OUTPUT_ROOT / name / 'outputs.jsonl' for name in VARIANTS.values()]
    assert all(path.exists() for path in sources), 'Reúne las cinco salidas antes de puntuar.'
    command = [sys.executable, str(D3 / 'eval/five_15.py'), 'score',
               '--output', str(OUTPUT_ROOT.parent / 'evaluation-15')]
    for name, path in zip(VARIANTS.values(), sources):
        command += ['--source', f'{name}={path}']
    subprocess.run(command, check=True)
else:
    print('Activa RUN_EVALUATE_15 cuando las cinco variantes tengan 15 casos.')


### Comparación pareada de los puntajes

El juez guarda primero los 75 juicios y sus cinco CSV. La celda siguiente calcula las diferencias sin usar GPU: lee esos juicios, verifica de nuevo FOM-5 v2 y compara los **mismos 15 IDs** con baseline y QLoRA archivados en la Entrega 2. El bootstrap pareado usa 10 000 remuestreos y semilla 42.

In [ ]:
RUN_COMPARE_15 = False
if RUN_COMPARE_15:
    from google.colab import drive
    drive.mount('/content/drive')
    sources = [OUTPUT_ROOT / name / 'outputs.jsonl' for name in VARIANTS.values()]
    command = [sys.executable, str(D3 / 'eval/five_15.py'), 'summarize',
               '--output', str(OUTPUT_ROOT.parent / 'evaluation-15')]
    for name, path in zip(VARIANTS.values(), sources):
        command += ['--source', f'{name}={path}']
    subprocess.run(command, check=True)
else:
    print('Activa RUN_COMPARE_15 después de completar los 75 juicios.')


In [ ]:
summary_path = OUTPUT_ROOT.parent / 'evaluation-15/summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    display(Markdown('### FOM-5 v2 · cinco variantes × 15 casos'))
    display(summary)
else:
    print('La evaluación de los 15 casos aún no está completa.')
